# 21 - Daughter hierarchy length `accdm_l_max` across masses

Notebook 19 found that the default hierarchy length 17 biases warm daughters at large f̃, and that
`l_max_ncdm` 70 was not converged everywhere (σ8_cb moves 0.4% between daughter 50 and 70 at 10¹¹ GeV,
f̃ = 0.5). `accdm_l_max` (`7154651a`) now sets the daughter alone. Which value does each mass need, with
which momentum grid, and how low can cold daughters go?

1. Warm convergence: `accdm_l_max` 17–100 against 140 at 10¹¹–10¹²·⁵ GeV, 50 bins/decade.
2. Grid × l_max at the hard warm points: 50–400 bins/decade × 50, 100, 140, against (400, 140).
3. Cold floor: `accdm_l_max` 4–17 against 35 at 10¹²·⁵–10¹⁸ GeV.
4. Cost: CPU time of the candidate settings.

**Result.** (pending)

Settings of nb19 (κ = 12.1, a_t = 0.133, strategy 5, s = 0.25, DE sink on, ΛCDM best fit and ω_dm,tot of
the 10¹⁴ GeV chain of 2026-10-07), neutrinos at `l_max_ncdm` 17, on the build with the flat-space ℓ-sampling
fix (`cbeb3e86`), so comparisons resolve below the old 0.04 jumps. Δχ²: `accdm_nb.compare`; target 0.01.
Runs come from the cluster: `cluster_scripts/gen_nb21_cache.py` defines the grid (imported here) and
writes `accDM_scans/nb21_cache/` (see `cluster_scripts/README.md`); this notebook only reads them.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from accdm_nb import RunCache, chi2_parts, compare, ok, show, status, plot_style
from cluster_scripts.gen_nb21_cache import (NULL, WARM, F_WARM, L_WARM, L_REF, HARD, BINS, L_GRID, COLD, F_COLD,
                                            L_COLD, L_COLD_REF, TIME_RUNS, params, run_points)

plot_style()
TARGET, LOOSE = 0.01, 0.1
SCANS = Path('accDM_scans/nb21_cache')
cache = RunCache('21_cache.pkl', also=sorted(SCANS.glob('*.pkl')))
missing = [pt for pt in run_points() if cache._get(cache.key(params(*pt))) is None]
assert not missing, '{} runs missing, e.g. {}: run gen_nb21_cache.py on the cluster'.format(len(missing), missing[:3])


def res(lm, ft, L, bins=50):
    return cache(params(lm, ft, L, bins))


null = res(*NULL)
print('{} runs read; failed:'.format(len(run_points())),
      [(pt, status(res(*pt))) for pt in run_points() if not ok(res(*pt))] or 'none')

## 1. Warm convergence

Δχ² of each `accdm_l_max` against 140 at 50 bins/decade; *signal* is the reference against f̃ = 10⁻⁶.

In [ ]:
rows = []
for lm in WARM:
    for ft in F_WARM:
        ref = res(lm, ft, L_REF)
        rows.append({'log10 m': lm, 'f̃': ft, 'signal': chi2_parts(ref, null)['total'],
                     **{'Δχ² {}'.format(L): compare(res(lm, ft, L), ref)['total'] for L in L_WARM},
                     'Δσ8_cb 17': compare(res(lm, ft, 17), ref)['dsigma8'],
                     'Δσ8_cb 70': compare(res(lm, ft, 70), ref)['dsigma8']})
t1 = pd.DataFrame(rows).set_index(['log10 m', 'f̃'])
show(t1, {'Δσ8_cb 17': '{:+.1e}', 'Δσ8_cb 70': '{:+.1e}'}, default='{:.2e}')

In [ ]:
fig, axes = plt.subplots(1, len(WARM), figsize=(16, 3.6), sharey=True, constrained_layout=True)
for ax, lm in zip(axes, WARM):
    for c, ft in zip(plt.cm.viridis(np.linspace(0, 0.9, len(F_WARM))), F_WARM):
        ax.loglog(L_WARM, [t1.loc[(lm, ft), 'Δχ² {}'.format(L)] for L in L_WARM], 'o-', color=c, ms=4,
                  label='f̃ = {:g}'.format(ft))
    ax.axhline(TARGET, color='k', ls='--', lw=0.8)
    ax.set_title('log10 m = {:g}, vs accdm_l_max {}'.format(lm, L_REF), fontsize=10)
    ax.set_xlabel('accdm_l_max')
axes[0].set_ylabel('Δχ²')
axes[0].legend(fontsize=7)
plt.show()

## 2. Grid × l_max at the hard warm points

Δχ² of (bins/decade, `accdm_l_max`) against (400, 140). If the errors are independent, the l_max error
(rows at fixed bins) is the same at every grid and the grid error (columns at fixed l_max) the same at every
l_max.

In [ ]:
rows = []
for lm, ft in HARD:
    ref = res(lm, ft, 140, 400)
    for b in BINS:
        rows.append({'point': '{:g}, {:g}'.format(lm, ft), 'bins/dec': b,
                     **{'l_max {}'.format(L): compare(res(lm, ft, L, b), ref)['total'] for L in L_GRID}})
t2 = pd.DataFrame(rows).set_index(['point', 'bins/dec'])
show(t2, default='{:.2e}')

## 3. Cold floor

Δχ² of short daughter hierarchies against `accdm_l_max` 35 at 50 bins/decade.

In [ ]:
rows = []
for lm in COLD:
    for ft in F_COLD:
        ref = res(lm, ft, L_COLD_REF)
        rows.append({'log10 m': lm, 'f̃': ft, 'signal': chi2_parts(ref, null)['total'],
                     **{'Δχ² {}'.format(L): compare(res(lm, ft, L), ref)['total'] for L in L_COLD}})
t3 = pd.DataFrame(rows).set_index(['log10 m', 'f̃'])
show(t3, default='{:.2e}')

## 4. Cost

CPU time of full runs (all threads, `time.process_time`) on a cluster node with 4 threads, twice each,
relative to the default (50 bins/decade, `accdm_l_max` 17) at the same point.

In [ ]:
cpu = {}
for f in sorted((SCANS/'cpu').glob('*.pkl')):
    cpu.update(pickle.loads(f.read_bytes()))
rows = [{'point': '{:g}, {:g}'.format(lm, ft), 'bins/dec': b, 'accdm_l_max': L, 'CPU [s]': np.mean(cpu[(lm, ft, b, L)]),
         'relative': np.mean(cpu[(lm, ft, b, L)])/np.mean(cpu[(lm, ft, 50, 17)])}
        for (lm, ft), runs in TIME_RUNS.items() for b, L in runs]
show(pd.DataFrame(rows).set_index(['point', 'bins/dec', 'accdm_l_max']), {'CPU [s]': '{:.0f}'}, default='{:.2f}')

## 5. Recommendation

Per mass, the smallest `accdm_l_max` whose Δχ² stays below 0.01 (and 0.1) at every f̃ tested, at 50
bins/decade (sections 1 and 3).

In [ ]:
def smallest(table, lm, ls, tol):
    for L in ls:
        if table.loc[lm]['Δχ² {}'.format(L)].max() <= tol:
            return L
    return np.nan


rows = [{'log10 m': lm, 'accdm_l_max for 0.01': smallest(t1, lm, L_WARM, TARGET),
         'for 0.1': smallest(t1, lm, L_WARM, LOOSE)} for lm in WARM]
rows += [{'log10 m': lm, 'accdm_l_max for 0.01': smallest(t3, lm, L_COLD, TARGET),
          'for 0.1': smallest(t3, lm, L_COLD, LOOSE)} for lm in COLD if lm not in WARM]
show(pd.DataFrame(rows).set_index('log10 m'), default='{:.0f}')